# 🕸️ Section 11: Simple LangGraph Workflow

LangGraph ka use cyclic aur stateful LLM workflows banane ke liye kiya jata hai.
Is notebook mein hum ek minimal, clean **StateGraph** create karenge jo user messages ko process karke response generate karta hai.

### 📦 1. Imports & Environment Setup
- **Purpose**: Master environment variables se API keys load karna aur LangGraph ke core components import karna.
- **Libraries**:
  - `dotenv`: `.env` file se credentials load karne ke liye.
  - `langchain.chat_models.init_chat_model`: Unified model loader.
  - `langgraph.graph`: `StateGraph`, `START`, `END` primitives.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chat_models import init_chat_model

print("Imports and environment loaded successfully.")

### 🤖 2. Model Initialization
- **Purpose**: Groq provider ke zariye high-speed chat model initialize karna.
- **Model**: `'openai/gpt-oss-20b'`
- **Temperature**: `0.7`

In [ ]:
model = init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')
print("Model ready:", model)

### 📋 3. State Schema Definition
- **Class**: `State(TypedDict)`
- **Field**: `messages: Annotated[list, add_messages]`
- **Explanation**:
  - `TypedDict` define karta hai ki state dictionary mein kaunse keys honge.
  - `add_messages` reducer function ensure karta hai ki naye messages purane messages mein append hon, na ki overwrite.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

### ⚙️ 4. Chatbot Node & Graph Compilation
- **Node**: `chatbot(state: State) -> dict`
- **Edges**:
  - `START ➔ chatbot`: Execution start hote hi chatbot node call hota hai.
  - `chatbot ➔ END`: Response generate hone ke baad workflow complete hota hai.
- **Compile**: `builder.compile()`

In [ ]:
def chatbot(state: State):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()
print("LangGraph workflow compiled successfully!")

### 🚀 5. Graph Invocation Test
- **Input**: `HumanMessage(content='What is LangGraph in 2 lines?')`
- **Method**: `graph.invoke()`
- **Output**: Final AI response display karna.

In [ ]:
result = graph.invoke({
    "messages": [HumanMessage(content="What is LangGraph in 2 lines?")]
})

print("AI Response:")
print(result["messages"][-1].content)